# Explaining a fit to a language model

A fitted boosting model is a set of numbers: an offset, the sequence of learners
the path selected, and one coefficient vector per learner. `llm_explain_prompt()`
writes those numbers as text a language model can read, and `llm_describe()`
sends that text to a chat endpoint and returns the reply.

The prompt is built from six sections, each of which `include=` can select:

`reading`
: How component-wise boosting produced the fit, stated in the terms the rest of
  the prompt uses: an offset plus one effect per learner, each effect the
  accumulated sum of its updates, each update scaled by the step length `nu`
  (the learning rate), and a learner never selected contributing exactly zero.

`facts`
: The `summary()` tables — family, `mstop`, `nu`, offset, risk at the offset and
  at `mstop` — plus what one unit of the additive predictor means. For a
  `Binomial` fit that unit is half a log-odds, and the section says so, because a
  reader told only "the effect is +1.4" cannot convert it.

`importance`
: Every base learner ranked by its share of the in-bag risk reduction (the
  reduction of the loss on the training rows), with the
  number of updates it took and the iteration it entered. Learners the path never
  chose are listed too, as contributing zero.

`effects`
: One block per learner: its basis and penalty, the covariate's observed range,
  the contribution at a set of covariate values, the contribution's span, and a
  one-line statement of the shape.

`caveats`
: The claims the fit does not support — no standard errors, importance shared
  between correlated covariates, shapes valid only over the observed range,
  everything conditional on `mstop`.

`task`
: The closing instruction, which `task=` replaces.

`llm_explain_prompt()` returns text locally for inspection.
`llm_describe()` sends that text to the configured chat endpoint.

In [1]:
import mboostrs as mb
from mboostrs.datasets import bodyfat

bf = bodyfat()
formula = (
    "DEXfat ~ bbs(age, df=4) + bbs(waistcirc, df=4) + bbs(hipcirc, df=4) + "
    "bbs(kneebreadth, df=4) + bbs(anthro3a, df=4)"
)
fit = mb.gamboost(formula, data=bf, control=mb.boost_control(mstop=200, nu=0.1))
fit

GAMBoostModel(formula='DEXfat ~ bbs(age, df=4) + bbs(waistcirc, df=4) + bbs(hipcirc, df=4) + bbs(kneebreadth, df=4) + bbs(anthro3a, df=4)', family=Gaussian, mstop=200)

## Learner importance ranking

Importance is the drop in in-bag risk (the loss summed over the training
rows) at each update, credited to the learner
that update chose, divided by the total over the path. It is attributed per
learner rather than per name, so the two learners of a decomposition such as
`bols(x) + bbs(x, df=1, center=True)` keep separate shares.

In [2]:
print(fit.llm_explain_prompt(include=("importance",)))

Base learners, ranked:
  hipcirc — importance 43.8% — 34 updates — entered at iteration 1
  anthro3a — importance 27.0% — 70 updates — entered at iteration 4
  waistcirc — importance 23.8% — 45 updates — entered at iteration 2
  kneebreadth — importance 5.4% — 42 updates — entered at iteration 14
  age — importance 0.1% — 9 updates — entered at iteration 52


## Effect blocks

A smooth learner in one covariate is evaluated on an equispaced grid of
`n_points` values, so its block is the effect itself rather than a summary of it.
Any other learner is evaluated at the training rows and condensed: a covariate
with few distinct values is reported at those values — which is what a factor, an
indicator, or a stump needs — and a continuous one is reported over equal-count
intervals, each carrying the mean contribution of its rows.

Contributions are rounded to the decimals the effect's own span supports: about
three significant digits of the span, so an effect moving by 0.004 keeps more
decimals than one moving by 40. The `Shape:` line reports monotonicity, or the
number of turning points and where the effect is highest and lowest, treating a
step under one per cent of the span as flat.

In [3]:
print(fit.llm_explain_prompt(include=("effects",), top=2, n_points=8))

Effects (2 of 5 selected learners, most important first):

Learner: hipcirc  [bbs: B-spline basis of degree 3 on 20 inner knots, df=4, second-difference penalty]
Selected in 34 of 200 updates (17.0%), first at iteration 1
Importance: 43.8% of total risk reduction
Covariate: hipcirc (observed range [88, 132])
Contribution: {"88": -7.69, "94.29": -4.63, "100.6": -1.27, "106.9": 1.36, "113.1": 3.79, "119.4": 5.56, "125.7": 6.28, "132": 6.2}
Contribution range: [-7.693, 6.28] (span 13.97)
Shape: monotone increasing over the observed range

Learner: anthro3a  [bbs: B-spline basis of degree 3 on 20 inner knots, df=4, second-difference penalty]
Selected in 70 of 200 updates (35.0%), first at iteration 4
Importance: 27.0% of total risk reduction
Covariate: anthro3a (observed range [2.4, 4.68])
Contribution: {"2.4": -9.38, "2.726": -7.47, "3.051": -5.6, "3.377": -3.68, "3.703": -1.13, "4.029": 0.64, "4.354": 3.46, "4.68": 9.43}
Contribution range: [-9.384, 9.431] (span 18.82)
Shape: monotone in

## Factors, interactions, and learners over two covariates

The learners in this section are described on
[the base-learner page](base_learners.ipynb). A `brandom` effect (a
ridge-penalized effect with one coefficient per level of a categorical column)
is keyed by level label rather than by the integer code the design uses, and
no shape is claimed for it, because levels have no order to be monotone in. A
learner reading two covariates (`bspatial`, a penalized spline surface; a
`%X%` row-wise tensor product of two smooth bases; a `btree` regression tree
over a pair) is reported as the mean contribution per cell of
an equal-count grid over both, with the term named for what the formula wrote
rather than for the derived column the tensor is fitted through.

In [4]:
import numpy as np

rng = np.random.default_rng(0)
n = 300
x1 = rng.uniform(0, 1, n)
x2 = rng.uniform(0, 1, n)
grade = np.array(["low", "mid", "high"])[rng.integers(0, 3, n)]
y = np.sin(6 * x1) * x2 + (grade == "high") * 0.8 + rng.normal(0, 0.2, n)
data = {"y": y, "x1": x1, "x2": x2, "grade": grade}

joint = mb.gamboost(
    "y ~ bbs(x1) %X% bbs(x2) + brandom(grade)",
    data=data,
    control=mb.boost_control(mstop=100, nu=0.1),
)
print(joint.llm_explain_prompt(include=("effects",), n_points=6, cells=4))

Effects (2 of 2 selected learners, most important first):

Learner: bbs(x1) %X% bbs(x2)  [row-wise tensor product of a bbs basis on x1 and a bbs basis on x2, penalized by the Kronecker sum of the two marginal penalties]
Selected in 68 of 100 updates (68.0%), first at iteration 1
Importance: 57.8% of total risk reduction
Covariates: x1 in [0.0003007, 0.9972], x2 in [0.004899, 0.9995]
Contribution range: [-0.9653, 0.8624] (span 1.828)
Mean contribution per cell of an equal-count grid; rows are x1, columns are x2:
x1 \ x2, (0.004899, 0.2655), (0.2655, 0.5317), (0.5317, 0.7273), (0.7273, 0.9995)
(0.0003007, 0.2995), 0.1558, 0.3217, 0.4194, 0.6682
(0.2995, 0.5609), 0.1556, 0.2994, 0.2758, 0.4
(0.5609, 0.8104), 0.02343, -0.2356, -0.4108, -0.6551
(0.8104, 0.9972), -0.03991, -0.2287, -0.4263, -0.638

Learner: brandom(grade)  [brandom: ridge-penalized effect over the levels of the column, df=4]
Selected in 32 of 100 updates (32.0%), first at iteration 3
Importance: 42.2% of total risk reduction

## Additional context: descriptions, cross-validation, stability selection

A fit records column names, not what the columns measure. `dataset_description=`
and `response_description=` supply that. `cv=` takes a `cvrisk` result so the
prompt can speak to the stopping iteration; without one the prompt says that
nothing in it indicates whether the fit stopped too early or too late.
`stabsel=` takes a stability-selection result (refits on random half-samples of the data, with the learners chosen most often kept) and adds, per learner, its
selection probability: the share of subsamples of the data on whose fit the
learner was selected.

In [5]:
cv = fit.cvrisk(folds=mb.cv_weights(len(bf["DEXfat"]), type="kfold", B=5, seed=1))
prompt = fit.llm_explain_prompt(
    dataset_description=(
        "Body fat of 71 healthy German women (Garcia et al. 2005). DEXfat is body "
        "fat measured by dual-energy X-ray absorptiometry, the reference method; "
        "the covariates are cheap anthropometric measurements in cm."
    ),
    response_description="DEXfat is body fat in kg.",
    cv=cv,
)
print(f"{len(prompt)} characters, roughly {len(prompt) // 4} tokens")

10296 characters, roughly 2574 tokens


## Sending the prompt: `llm_describe`

`llm_describe()` posts the prompt to an OpenAI-compatible chat-completions
endpoint and returns the reply. The endpoint defaults to NVIDIA's hosted
inference API and the model to `nvidia/nemotron-3-ultra-550b-a55b`; the key is
read from `MBOOSTRS_LLM_API_KEY`, `NVIDIA_API_KEY`, or `OPENAI_API_KEY`.
`base_url=` points
it at anything else that speaks the same protocol — a local NIM container, vLLM,
llama.cpp, Ollama, OpenAI — and `client=` uses a client object you already have
instead of the built-in request.

The prompt instructs the model to reply in Markdown: short headings or bold
labels per effect, bullets, and small tables where they help. Render the
returned text with a Markdown renderer rather than printing it as plain text.

```python
answer = fit.llm_describe(
    dataset_description="Body fat of 71 healthy German women; covariates in cm.",
    response_description="DEXfat is body fat in kg.",
    llm="nvidia/nemotron-3-ultra-550b-a55b",
)
```

`reasoning=True` runs three turns in one conversation instead of one: describe
the effects, then name what is surprising and what could produce it, then
compress both into `num_sentences`. The summary is the return value;
`return_conversation=True` returns every turn.

```python
answer = fit.llm_describe(reasoning=True, num_sentences=5)
```

To drive the turns yourself, `output="turns"` returns the system message and the
three user turns without sending them; `output="messages"` returns the
system/user pair for the single-request form.

`llm_describe()` sends the prompt off the machine. The prompt contains
covariate names, observed ranges, and fitted effects: a summary of the training
data rather than its rows, which can still disclose information about small or
identifiable groups. Read the prompt before sending it anywhere.

## R interface

The R frontend has the same two functions, and the fixed sections are worded
identically, so a prompt built in either language reads the same to the model
receiving it. `llm_describe()` posts through the **curl** package when it is
installed and the `curl` command-line tool otherwise.

```r
fit <- gamboost(DEXfat ~ bbs(waistcirc, df = 4) + bbs(hipcirc, df = 4),
                data = bodyfat, control = boost_control(mstop = 200, nu = 0.1))

cat(llm_explain_prompt(fit, include = c("importance", "effects"), top = 2))

cat(llm_describe(fit, dataset_description = "Body fat of 71 women; cm."))
```

The R version describes `glmboost` and `gamboost` fits. The Python version
also describes location, scale and shape (LSS) models, `gamboostLSS` and
`glmboostLSS` fits, with one block per distribution parameter,
each labelled with that parameter's link.

## Limitations

The prompt reports fitted effects, but the generated explanation can assign
substantive meaning to sampling noise. For example, an effect selected in
9 of 200 updates and spanning 0.9 kg needs to be assessed against response
variability and resampling results. The prompt supplies its update count and
span; these alone do not determine whether the effect represents signal.

Importance attributes the risk reduction along one fitted path; it does not
measure a causal contribution. When two
covariates are strongly correlated, which of them collects the credit is decided
by which the first update happened to pick, and the prompt's caveats say so.

The prompt contains no standard errors or intervals. `confint()` can bootstrap
intervals for fitted effects, subject to its treatment of selection and tuning;
those results must be examined separately from the generated explanation.